# Notebook 4: Transformations vs Actions

## Goal

The goal of this notebook is to understand when Spark actually runs code.

Spark is lazily evaluated. This means Spark does not execute most DataFrame operations immediately.

Instead, Spark builds a logical plan.

Execution happens only when you call an action.

This is one of the most important Spark concepts to understand.

By the end of this notebook, you should understand the difference between:

1. Transformations
2. Actions
3. Lazy evaluation
4. Query plans
5. Spark execution triggers

## What You Will Learn

| Type | Examples |
|---|---|
| Transformations | `select`, `filter`, `withColumn`, `groupBy` |
| Actions | `show`, `count`, `collect`, `display`, `write` |

## Key Idea

Transformations define a new DataFrame.

Actions trigger Spark to actually compute a result.

For example:

```python
filtered_df = transactions_df.filter("amount > 100")
```
This line defines a transformation.
But this line triggers execution:
```
filtered_df.show()
```
Spark waits until an action is called before running the actual job.

In [0]:
# Import common Spark SQL functions and data types.

from pyspark.sql import functions as F
from pyspark.sql.types import (
   StructType,
   StructField,
   IntegerType,
   StringType,
   DoubleType,
   TimestampType
)
from datetime import datetime

In [0]:
# Confirm that SparkSession exists.
# In Databricks, spark is usually available automatically.

spark

In [0]:
# Print the Spark version.

print("Spark version:", spark.version)

Spark version: 4.1.0


# Section 2: Create a Practice DataFrame

## 1. Create a Practice Transaction DataFrame

We will create a small retail transaction DataFrame.

Each row represents one transaction.

This DataFrame will let us practice transformations and actions.

In [0]:
# Create sample retail transaction data.
# Each tuple represents one transaction.

transaction_data = [
   (1001, "Store_001", "Customer_001", 249.99, "Electronics", "Credit Card", datetime(2026, 5, 1, 9, 15, 0)),
   (1002, "Store_001", "Customer_002", 34.50, "Grocery", "Cash", datetime(2026, 5, 1, 10, 5, 0)),
   (1003, "Store_002", "Customer_003", 89.99, "Clothing", "Debit Card", datetime(2026, 5, 1, 11, 30, 0)),
   (1004, "Store_002", "Customer_001", 599.99, "Electronics", "Credit Card", datetime(2026, 5, 2, 14, 45, 0)),
   (1005, "Store_003", "Customer_004", 22.25, "Grocery", "Cash", datetime(2026, 5, 2, 16, 20, 0)),
   (1006, "Store_003", "Customer_005", 129.99, "Home Goods", "Credit Card", datetime(2026, 5, 3, 13, 10, 0)),
   (1007, "Store_001", "Customer_006", 14.99, "Grocery", "Debit Card", datetime(2026, 5, 3, 17, 55, 0)),
   (1008, "Store_002", "Customer_007", 799.99, "Electronics", "Credit Card", datetime(2026, 5, 4, 12, 0, 0)),
   (1009, "Store_003", "Customer_008", 45.00, "Clothing", "Cash", datetime(2026, 5, 4, 15, 35, 0)),
   (1010, "Store_001", "Customer_009", 199.99, "Home Goods", "Credit Card", datetime(2026, 5, 5, 18, 10, 0)),
   (1011, "Store_004", "Customer_010", 1099.99, "Electronics", "Credit Card", datetime(2026, 5, 5, 19, 25, 0)),
   (1012, "Store_004", "Customer_011", 64.75, "Grocery", "Debit Card", datetime(2026, 5, 6, 8, 45, 0))
]

In [0]:
# Define a schema for the transaction data.

transaction_schema = StructType([
   StructField("transaction_id", IntegerType(), nullable=False),
   StructField("store_id", StringType(), nullable=False),
   StructField("customer_id", StringType(), nullable=False),
   StructField("amount", DoubleType(), nullable=False),
   StructField("category", StringType(), nullable=True),
   StructField("payment_method", StringType(), nullable=True),
   StructField("timestamp", TimestampType(), nullable=False)
])

In [0]:
# Create the Spark DataFrame.
# This creates a DataFrame object from local Python data.

transactions_df = spark.createDataFrame(
   transaction_data,
   schema=transaction_schema
)

In [0]:
# display() is an action.
# It triggers Spark to compute and display the DataFrame.

display(transactions_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,Electronics,Credit Card,2026-05-01T09:15:00.000Z
1002,Store_001,Customer_002,34.5,Grocery,Cash,2026-05-01T10:05:00.000Z
1003,Store_002,Customer_003,89.99,Clothing,Debit Card,2026-05-01T11:30:00.000Z
1004,Store_002,Customer_001,599.99,Electronics,Credit Card,2026-05-02T14:45:00.000Z
1005,Store_003,Customer_004,22.25,Grocery,Cash,2026-05-02T16:20:00.000Z
1006,Store_003,Customer_005,129.99,Home Goods,Credit Card,2026-05-03T13:10:00.000Z
1007,Store_001,Customer_006,14.99,Grocery,Debit Card,2026-05-03T17:55:00.000Z
1008,Store_002,Customer_007,799.99,Electronics,Credit Card,2026-05-04T12:00:00.000Z
1009,Store_003,Customer_008,45.0,Clothing,Cash,2026-05-04T15:35:00.000Z
1010,Store_001,Customer_009,199.99,Home Goods,Credit Card,2026-05-05T18:10:00.000Z


In [0]:
# Inspect the schema.
# printSchema() displays the structure of the DataFrame.

transactions_df.printSchema()

root
 |-- transaction_id: integer (nullable = false)
 |-- store_id: string (nullable = false)
 |-- customer_id: string (nullable = false)
 |-- amount: double (nullable = false)
 |-- category: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- timestamp: timestamp (nullable = false)



# Section 3: What Is a Transformation?

## 2. What Is a Transformation?

A transformation creates a new DataFrame from an existing DataFrame.

Examples include:

| Transformation | What It Does |
|---|---|
| `select()` | Chooses columns |
| `filter()` | Keeps rows matching a condition |
| `where()` | Same idea as `filter()` |
| `withColumn()` | Adds or replaces a column |
| `groupBy()` | Defines grouped calculations |
| `orderBy()` | Sorts rows |
| `drop()` | Removes columns |

Transformations are lazy.

That means Spark records the operation, but does not immediately compute the result.

In [0]:
# This is a transformation.
# It defines a new DataFrame with only selected columns.
# Spark does not immediately execute a job just because we define this variable.

selected_columns_df = transactions_df.select(
   "transaction_id",
   "store_id",
   "amount",
   "category"
)

# At this point, Spark has created a logical plan.
# It has not displayed or collected any data yet.

In [0]:
# This is another transformation.
# It defines a filtered DataFrame.
# Spark still does not execute the computation yet.

high_value_df = selected_columns_df.filter(
   F.col("amount") > 100
)

In [0]:
# This is another transformation.
# It defines a new column called high_value_flag.
# Spark still does not execute yet.

high_value_with_flag_df = high_value_df.withColumn(
   "high_value_flag",
   F.lit(True)
)

In [0]:
# Check the type of the object.
# It is a Spark DataFrame.

type(high_value_with_flag_df)

pyspark.sql.connect.dataframe.DataFrame

## Important Observation

So far, we have defined several transformations:

```python
select()
filter()
withColumn()
```

But Spark has not needed to show us any rows yet.
It has only built a plan.
This is lazy evaluation.


# Section 4: What Is an Action?

## 3. What Is an Action?

An action triggers Spark to execute the plan and return a result.

Examples include:

| Action | What It Does |
|---|---|
| `show()` | Prints rows to the notebook output |
| `display()` | Displays rows interactively in Databricks |
| `count()` | Counts rows |
| `collect()` | Brings all rows back to the driver |
| `first()` | Returns the first row |
| `take(n)` | Returns first n rows |
| `write` | Writes data to storage |

Actions are where Spark actually runs computation.

In [0]:
# show() is an action.
# This triggers Spark to execute the transformations above.

high_value_with_flag_df.show()

+--------------+---------+-------+-----------+---------------+
|transaction_id| store_id| amount|   category|high_value_flag|
+--------------+---------+-------+-----------+---------------+
|          1001|Store_001| 249.99|Electronics|           true|
|          1004|Store_002| 599.99|Electronics|           true|
|          1006|Store_003| 129.99| Home Goods|           true|
|          1008|Store_002| 799.99|Electronics|           true|
|          1010|Store_001| 199.99| Home Goods|           true|
|          1011|Store_004|1099.99|Electronics|           true|
+--------------+---------+-------+-----------+---------------+



In [0]:
# count() is also an action.
# Spark must compute how many rows are in the DataFrame.

num_high_value_rows = high_value_with_flag_df.count()

print("Number of high-value rows:", num_high_value_rows)

Number of high-value rows: 6


In [0]:
# first() is an action.
# It returns the first row from the DataFrame.

first_row = high_value_with_flag_df.first()

first_row

Row(transaction_id=1001, store_id='Store_001', amount=249.99, category='Electronics', high_value_flag=True)

In [0]:
# take(3) is an action.
# It returns the first 3 rows as a Python list of Row objects.

first_three_rows = high_value_with_flag_df.take(3)

first_three_rows

[Row(transaction_id=1001, store_id='Store_001', amount=249.99, category='Electronics', high_value_flag=True),
 Row(transaction_id=1004, store_id='Store_002', amount=599.99, category='Electronics', high_value_flag=True),
 Row(transaction_id=1006, store_id='Store_003', amount=129.99, category='Home Goods', high_value_flag=True)]

# Section 5: Be Careful with collect()

## 4. Be Careful with collect()

The `collect()` action brings all rows from the Spark DataFrame back to the driver as a Python list.

This is okay for tiny practice datasets.

But on large datasets, `collect()` can crash your driver because it tries to move all distributed data into local memory.

Use `collect()` carefully.

For learning, it is fine.

For large production datasets, avoid it unless you are absolutely sure the result is small.

In [0]:
# collect() is an action.
# It brings all rows back to the driver as a Python list.
# This is safe here because our dataset is tiny.

rows_as_python_list = high_value_with_flag_df.collect()

rows_as_python_list

[Row(transaction_id=1001, store_id='Store_001', amount=249.99, category='Electronics', high_value_flag=True),
 Row(transaction_id=1004, store_id='Store_002', amount=599.99, category='Electronics', high_value_flag=True),
 Row(transaction_id=1006, store_id='Store_003', amount=129.99, category='Home Goods', high_value_flag=True),
 Row(transaction_id=1008, store_id='Store_002', amount=799.99, category='Electronics', high_value_flag=True),
 Row(transaction_id=1010, store_id='Store_001', amount=199.99, category='Home Goods', high_value_flag=True),
 Row(transaction_id=1011, store_id='Store_004', amount=1099.99, category='Electronics', high_value_flag=True)]

In [0]:
# We can loop through the collected rows because they are now local Python objects.
# Again, this is only recommended for small results.

for row in rows_as_python_list:
   print(row)

Row(transaction_id=1001, store_id='Store_001', amount=249.99, category='Electronics', high_value_flag=True)
Row(transaction_id=1004, store_id='Store_002', amount=599.99, category='Electronics', high_value_flag=True)
Row(transaction_id=1006, store_id='Store_003', amount=129.99, category='Home Goods', high_value_flag=True)
Row(transaction_id=1008, store_id='Store_002', amount=799.99, category='Electronics', high_value_flag=True)
Row(transaction_id=1010, store_id='Store_001', amount=199.99, category='Home Goods', high_value_flag=True)
Row(transaction_id=1011, store_id='Store_004', amount=1099.99, category='Electronics', high_value_flag=True)


# Section 6: Lazy Evaluation Demonstration

## 5. Lazy Evaluation Demonstration

Let's build a longer transformation chain.

We will:

1. Select useful columns
2. Filter for transactions over 50 dollars
3. Add a transaction size label
4. Add a transaction month
5. Sort by amount

None of these transformations will execute until we call an action.

In [0]:
# Build a chain of transformations.
# This defines the logic, but does not execute it yet.

transformed_transactions_df = (
   transactions_df
   .select(
       "transaction_id",
       "store_id",
       "customer_id",
       "amount",
       "category",
       "payment_method",
       "timestamp"
   )
   .filter(F.col("amount") > 50)
   .withColumn(
       "transaction_size",
       F.when(F.col("amount") >= 500, "Large")
        .when(F.col("amount") >= 100, "Medium")
        .otherwise("Small")
   )
   .withColumn(
       "transaction_month",
       F.month("timestamp")
   )
   .orderBy(F.col("amount").desc())
)

# Spark has now built a plan.
# It has not executed the full computation yet.

In [0]:
# This action triggers Spark to execute the transformation chain.

display(transformed_transactions_df)

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp,transaction_size,transaction_month
1011,Store_004,Customer_010,1099.99,Electronics,Credit Card,2026-05-05T19:25:00.000Z,Large,5
1008,Store_002,Customer_007,799.99,Electronics,Credit Card,2026-05-04T12:00:00.000Z,Large,5
1004,Store_002,Customer_001,599.99,Electronics,Credit Card,2026-05-02T14:45:00.000Z,Large,5
1001,Store_001,Customer_001,249.99,Electronics,Credit Card,2026-05-01T09:15:00.000Z,Medium,5
1010,Store_001,Customer_009,199.99,Home Goods,Credit Card,2026-05-05T18:10:00.000Z,Medium,5
1006,Store_003,Customer_005,129.99,Home Goods,Credit Card,2026-05-03T13:10:00.000Z,Medium,5
1003,Store_002,Customer_003,89.99,Clothing,Debit Card,2026-05-01T11:30:00.000Z,Small,5
1012,Store_004,Customer_011,64.75,Grocery,Debit Card,2026-05-06T08:45:00.000Z,Small,5


# Section 7: Inspect the Query Plan with explain()


## 6. Inspect the Query Plan with explain()

The `explain()` method shows Spark's execution plan.

This helps us understand what Spark plans to do.

There are multiple plan stages Spark may show, such as:

- Parsed logical plan
- Analyzed logical plan
- Optimized logical plan
- Physical plan

You do not need to master all of these yet.

For now, remember:

> Spark transforms your DataFrame operations into an optimized execution plan.

In [0]:
# Show the basic physical plan.

transformed_transactions_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [amount#10975 DESC NULLS LAST]
         +- PhotonRowToColumnar
            +- LocalTableScan [transaction_id#10972, store_id#10973, customer_id#10974, amount#10975, category#10976, payment_method#10977, timestamp#10978, transaction_size#11028, transaction_month#11030]


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
# Show a more detailed plan.
# This can be verbose, but it helps you see more of Spark's planning process.

transformed_transactions_df.explain(mode="extended")

== Parsed Logical Plan ==
'Sort ['amount DESC NULLS LAST], true
+- Project [transaction_id#10972, store_id#10973, customer_id#10974, amount#10975, category#10976, payment_method#10977, timestamp#10978, transaction_size#11028, month(cast(timestamp#10978 as date)) AS transaction_month#11030]
   +- Project [transaction_id#10972, store_id#10973, customer_id#10974, amount#10975, category#10976, payment_method#10977, timestamp#10978, CASE WHEN (amount#10975 >= cast(500 as double)) THEN Large WHEN (amount#10975 >= cast(100 as double)) THEN Medium ELSE Small END AS transaction_size#11028]
      +- Filter (amount#10975 > cast(50 as double))
         +- Project [transaction_id#10972, store_id#10973, customer_id#10974, amount#10975, category#10976, payment_method#10977, timestamp#10978]
            +- LocalRelation [transaction_id#10972, store_id#10973, customer_id#10974, amount#10975, category#10976, payment_method#10977, timestamp#10978]

== Analyzed Logical Plan ==
transaction_id: int, store_i

## How to Read This at a Beginner Level

When you look at the `explain()` output, do not worry about understanding every detail.

For now, look for clues like:

| Term | Beginner Meaning |
|---|---|
| `Project` | Spark is selecting or creating columns |
| `Filter` | Spark is filtering rows |
| `Sort` | Spark is ordering rows |
| `Aggregate` | Spark is grouping and summarizing |
| `Exchange` | Spark may be moving data across partitions |
| `Scan` | Spark is reading data |

Later, when we study Spark performance, we will come back to these ideas.

# Section 8: groupBy as a Transformation

## 7. Is groupBy an Action?

This is a common beginner question.

`groupBy()` by itself is not an action.

It defines grouped data.

When combined with aggregation methods like `agg()`, it creates a new DataFrame.

But Spark still does not execute until you call an action like `show()`, `display()`, or `count()`.


In [0]:
# Define an aggregation.
# groupBy() and agg() define a transformation that creates a new DataFrame.
# This does not execute yet.

sales_by_store_df = (
   transactions_df
   .groupBy("store_id")
   .agg(
       F.sum("amount").alias("total_sales"),
       F.avg("amount").alias("avg_transaction_amount"),
       F.count("*").alias("num_transactions")
   )
   .orderBy(F.col("total_sales").desc())
)

In [0]:
# This action triggers Spark to compute the grouped result.

display(sales_by_store_df)

store_id,total_sales,avg_transaction_amount,num_transactions
Store_002,1489.97,496.6566666666667,3
Store_004,1164.74,582.37,2
Store_001,499.47,124.8675,4
Store_003,197.24,65.74666666666667,3


In [0]:
# Inspect the plan for the grouped calculation.

sales_by_store_df.explain(mode="extended")

== Parsed Logical Plan ==
'Sort ['total_sales DESC NULLS LAST], true
+- 'Aggregate ['store_id], ['store_id, 'sum('amount) AS total_sales#11038, 'avg('amount) AS avg_transaction_amount#11039, 'count(*) AS num_transactions#11040]
   +- LocalRelation [transaction_id#10972, store_id#10973, customer_id#10974, amount#10975, category#10976, payment_method#10977, timestamp#10978]

== Analyzed Logical Plan ==
store_id: string, total_sales: double, avg_transaction_amount: double, num_transactions: bigint
Sort [total_sales#11038 DESC NULLS LAST], true
+- Aggregate [store_id#10973], [store_id#10973, sum(amount#10975) AS total_sales#11038, avg(amount#10975) AS avg_transaction_amount#11039, count(1) AS num_transactions#11040L]
   +- LocalRelation [transaction_id#10972, store_id#10973, customer_id#10974, amount#10975, category#10976, payment_method#10977, timestamp#10978]

== Optimized Logical Plan ==
Sort [total_sales#11038 DESC NULLS LAST], true
+- Aggregate [store_id#10973], [store_id#10973, sum(a

## Important Note About Aggregations

Aggregations often require Spark to move data around.

For example, if rows for the same `store_id` are spread across different partitions, Spark may need to bring those rows together to compute total sales by store.

This movement of data is called a shuffle.

Shuffles are important for performance, but we will study them more deeply later.

# Section 9: Actions Compared

## 8. Common Actions Compared

Let's compare several actions.

Each of these triggers Spark execution, but they return different kinds of results.

In [0]:
# show() prints rows in a text table.
# It is useful for quick inspection.

transactions_df.show(5)

+--------------+---------+------------+------+-----------+--------------+-------------------+
|transaction_id| store_id| customer_id|amount|   category|payment_method|          timestamp|
+--------------+---------+------------+------+-----------+--------------+-------------------+
|          1001|Store_001|Customer_001|249.99|Electronics|   Credit Card|2026-05-01 09:15:00|
|          1002|Store_001|Customer_002|  34.5|    Grocery|          Cash|2026-05-01 10:05:00|
|          1003|Store_002|Customer_003| 89.99|   Clothing|    Debit Card|2026-05-01 11:30:00|
|          1004|Store_002|Customer_001|599.99|Electronics|   Credit Card|2026-05-02 14:45:00|
|          1005|Store_003|Customer_004| 22.25|    Grocery|          Cash|2026-05-02 16:20:00|
+--------------+---------+------------+------+-----------+--------------+-------------------+
only showing top 5 rows


In [0]:
# display() shows an interactive Databricks table.
# It is useful for notebook exploration.

display(transactions_df.limit(5))

transaction_id,store_id,customer_id,amount,category,payment_method,timestamp
1001,Store_001,Customer_001,249.99,Electronics,Credit Card,2026-05-01T09:15:00.000Z
1002,Store_001,Customer_002,34.5,Grocery,Cash,2026-05-01T10:05:00.000Z
1003,Store_002,Customer_003,89.99,Clothing,Debit Card,2026-05-01T11:30:00.000Z
1004,Store_002,Customer_001,599.99,Electronics,Credit Card,2026-05-02T14:45:00.000Z
1005,Store_003,Customer_004,22.25,Grocery,Cash,2026-05-02T16:20:00.000Z


In [0]:
# count() returns a number.
# It requires Spark to count the rows.

transactions_df.count()

12

In [0]:
# first() returns the first Row.

transactions_df.first()

Row(transaction_id=1001, store_id='Store_001', customer_id='Customer_001', amount=249.99, category='Electronics', payment_method='Credit Card', timestamp=datetime.datetime(2026, 5, 1, 9, 15))

In [0]:
# take(2) returns a list containing the first 2 Row objects.

transactions_df.take(2)

[Row(transaction_id=1001, store_id='Store_001', customer_id='Customer_001', amount=249.99, category='Electronics', payment_method='Credit Card', timestamp=datetime.datetime(2026, 5, 1, 9, 15)),
 Row(transaction_id=1002, store_id='Store_001', customer_id='Customer_002', amount=34.5, category='Grocery', payment_method='Cash', timestamp=datetime.datetime(2026, 5, 1, 10, 5))]

In [0]:
# collect() returns all rows as a Python list.
# Use carefully on real datasets.

transactions_df.collect()

[Row(transaction_id=1001, store_id='Store_001', customer_id='Customer_001', amount=249.99, category='Electronics', payment_method='Credit Card', timestamp=datetime.datetime(2026, 5, 1, 9, 15)),
 Row(transaction_id=1002, store_id='Store_001', customer_id='Customer_002', amount=34.5, category='Grocery', payment_method='Cash', timestamp=datetime.datetime(2026, 5, 1, 10, 5)),
 Row(transaction_id=1003, store_id='Store_002', customer_id='Customer_003', amount=89.99, category='Clothing', payment_method='Debit Card', timestamp=datetime.datetime(2026, 5, 1, 11, 30)),
 Row(transaction_id=1004, store_id='Store_002', customer_id='Customer_001', amount=599.99, category='Electronics', payment_method='Credit Card', timestamp=datetime.datetime(2026, 5, 2, 14, 45)),
 Row(transaction_id=1005, store_id='Store_003', customer_id='Customer_004', amount=22.25, category='Grocery', payment_method='Cash', timestamp=datetime.datetime(2026, 5, 2, 16, 20)),
 Row(transaction_id=1006, store_id='Store_003', customer_

# Section 10: Writing Data Is Also an Action

## 9. Writing Data Is Also an Action

Writing a DataFrame to storage is also an action.

For example:

```python
df.write.mode("overwrite").format("delta").saveAsTable("my_table")
```
This triggers Spark to compute the DataFrame and write the result.
In this beginner notebook, we will not write a permanent table yet unless you choose to uncomment the example below.
You will practice writing data more deeply in a later notebook.

In [0]:
# Optional example only.
# Uncomment this later if you want to practice writing a table.
# This may create a table in your Databricks environment.

transformed_transactions_df.write.mode("overwrite").format("delta").saveAsTable("transformed_transactions_demo")

# Section 11: Mini-Project

# Mini-Project: Build a Chain of Transformations and Inspect the Query Plan

In this mini-project, we will build a small Spark pipeline.

The pipeline will:

1. Start with raw transaction data
2. Select useful columns
3. Filter out low-value transactions
4. Add a risk or priority label
5. Create a store-level summary
6. Sort stores by total sales
7. Inspect the query plan with `explain()`

The goal is to understand that Spark builds a plan first and executes only when we call an action.

In [0]:
# Step 1: Select useful columns.
# This is a transformation.

mini_project_step_1_df = transactions_df.select(
   "transaction_id",
   "store_id",
   "customer_id",
   "amount",
   "category",
   "payment_method",
   "timestamp"
)

In [0]:
# Step 2: Filter out low-value transactions.
# This is also a transformation.

mini_project_step_2_df = mini_project_step_1_df.filter(
   F.col("amount") >= 50
)

In [0]:
# Step 3: Add a priority label.
# This is another transformation.

mini_project_step_3_df = mini_project_step_2_df.withColumn(
   "priority_label",
   F.when(F.col("amount") >= 500, "High Priority")
    .when(F.col("amount") >= 100, "Medium Priority")
    .otherwise("Low Priority")
)

In [0]:
# Step 4: Add transaction date fields.
# This is another transformation.

mini_project_step_4_df = (
   mini_project_step_3_df
   .withColumn("transaction_year", F.year("timestamp"))
   .withColumn("transaction_month", F.month("timestamp"))
   .withColumn("transaction_day", F.dayofmonth("timestamp"))
)

In [0]:
# Step 5: Create a store-level summary.
# groupBy() and agg() define a grouped transformation.

mini_project_summary_df = (
   mini_project_step_4_df
   .groupBy("store_id")
   .agg(
       F.sum("amount").alias("total_sales"),
       F.avg("amount").alias("avg_transaction_amount"),
       F.count("*").alias("num_transactions"),
       F.max("amount").alias("max_transaction_amount")
   )
   .orderBy(F.col("total_sales").desc())
)

In [0]:
# At this point, we have defined the full transformation chain.
# We have not displayed the summary yet.
# Now we inspect the query plan.

mini_project_summary_df.explain(mode="extended")

== Parsed Logical Plan ==
'Sort ['total_sales DESC NULLS LAST], true
+- 'Aggregate ['store_id], ['store_id, 'sum('amount) AS total_sales#11720, 'avg('amount) AS avg_transaction_amount#11721, 'count(*) AS num_transactions#11722, 'max('amount) AS max_transaction_amount#11723]
   +- Project [transaction_id#10972, store_id#10973, customer_id#10974, amount#10975, category#10976, payment_method#10977, timestamp#10978, priority_label#11711, transaction_year#11714, transaction_month#11716, dayofmonth(cast(timestamp#10978 as date)) AS transaction_day#11718]
      +- Project [transaction_id#10972, store_id#10973, customer_id#10974, amount#10975, category#10976, payment_method#10977, timestamp#10978, priority_label#11711, transaction_year#11714, month(cast(timestamp#10978 as date)) AS transaction_month#11716]
         +- Project [transaction_id#10972, store_id#10973, customer_id#10974, amount#10975, category#10976, payment_method#10977, timestamp#10978, priority_label#11711, year(cast(timestamp#1

In [0]:
# This action triggers Spark to execute the mini-project pipeline.

display(mini_project_summary_df)

store_id,total_sales,avg_transaction_amount,num_transactions,max_transaction_amount
Store_002,1489.97,496.6566666666667,3,799.99
Store_004,1164.74,582.37,2,1099.99
Store_001,449.98,224.99,2,249.99
Store_003,129.99,129.99,1,129.99


## Mini-Project Reflection

The mini-project followed this pattern:

```text
Original DataFrame
   ↓
select()
   ↓
filter()
   ↓
withColumn()
   ↓
withColumn()
   ↓
groupBy().agg()
   ↓
orderBy()
   ↓
explain()
   ↓
display()
```

Most of the steps were transformations.
The actual computation was triggered when we called an action like display().
This is how many Spark pipelines work in practice.

# Section 12: Practice Exercises


# Practice Exercises

Try these on your own.

## Exercise 1

Create a transformation that selects only:

- transaction_id
- store_id
- amount

Do not call `show()` or `display()` immediately.

Then check the type of the new object.

## Exercise 2

Create a transformation that filters for transactions where `category = "Electronics"`.

Then trigger execution with `show()`.

## Exercise 3

Create a transformation that adds a new column called `is_large_transaction`.

The column should be:

- `True` if amount is greater than or equal to 500
- `False` otherwise

Then display the result.

## Exercise 4

Create a grouped DataFrame that calculates total sales by category.

Before displaying it, call `explain()`.

Then call `display()`.

## Exercise 5

Use `count()` to count how many transactions are greater than 100 dollars.

## Exercise 6

Use `collect()` on a DataFrame with only 2 rows.

Why would this be dangerous on a very large DataFrame?

# Section 14: Common Mistakes

## Mistake 1: Thinking Every Line Executes Immediately

In normal Python, a line of code usually executes immediately.

In Spark, transformations are lazy.

This line does not immediately process all the data:

```python
filtered_df = df.filter(F.col("amount") > 100)
```

It defines a plan.

## Mistake 2: Using collect() Too Freely
This can be dangerous:
df.collect()
If df is huge, Spark will try to bring all rows back to the driver.
That can crash your notebook or cluster.

## Mistake 3: Forgetting That display() Is an Action
In Databricks, display(df) triggers computation.
It is not just a passive preview.

## Mistake 4: Not Using explain()
When learning Spark, explain() is useful because it shows that Spark builds a plan.
You do not need to fully understand every line yet, but you should get used to looking at it.

## Mistake 5: Assuming groupBy() Immediately Computes Results
groupBy() defines grouped logic.
The computation happens when an action is called.


# Section 15: Key Takeaways

In this notebook, you learned the difference between transformations and actions.

The most important ideas are:

1. Transformations define new DataFrames.
2. Actions trigger Spark execution.
3. Spark is lazily evaluated.
4. Spark builds a logical plan before executing code.
5. `select`, `filter`, `withColumn`, and `groupBy` are transformations.
6. `show`, `count`, `collect`, `display`, and `write` are actions.
7. `collect()` should be used carefully.
8. `explain()` lets you inspect Spark's query plan.
9. Spark does not simply run every DataFrame line immediately.
10. Spark optimizes a plan and executes it when needed.

The core pattern is:

```text
Define transformations
   ↓
Build logical plan
   ↓
Call an action
   ↓
Spark executes the plan
```